In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q11/annotated/checkpoints/pre_cell_3.pickle

trying: ['tpch_parent']
me:  0
trying: ['DATA_ROOT']
me:  0
trying: ['load_partsupp']
me:  1
trying: ['load_nation']
me:  3
trying: ['supplier']


me:  2
trying: ['nation']
me:  3
trying: ['STORAGE_OPTS']
me:  0
trying: ['partsupp']
me:  1
trying: ['load_supplier']
me:  2
trying: ['pd']
me:  0


Declaring variable tpch_parent
Declaring variable DATA_ROOT
Declaring variable STORAGE_OPTS
Declaring variable pd
Declaring variable load_partsupp
Declaring variable partsupp
Declaring variable supplier
Declaring variable load_supplier
Declaring variable load_nation
Declaring variable nation


In [4]:
%%RecordEvent
%%time
### cell 3 ###

# Compute total cost and select required columns in one step
ps = partsupp.assign(
    TOTAL_COST=partsupp.PS_SUPPLYCOST * partsupp.PS_AVAILQTY
)[["PS_PARTKEY", "PS_SUPPKEY", "TOTAL_COST"]]

# Filter supplier and nation tables
sup = supplier[["S_SUPPKEY", "S_NATIONKEY"]]
nat = nation[nation.N_NAME == "GERMANY"][["N_NATIONKEY"]]

# Join partsupp → supplier → nation, then keep only PS_PARTKEY and TOTAL_COST
# Note: specify left_on and right_on for the first merge
df = (
    ps
    .merge(sup, left_on="PS_SUPPKEY", right_on="S_SUPPKEY", how="inner")
    .merge(nat, left_on="S_NATIONKEY", right_on="N_NATIONKEY", how="inner")
    [["PS_PARTKEY", "TOTAL_COST"]]
)

# Compute threshold and aggregate per part
threshold = df.TOTAL_COST.sum() * 0.0001

total = (
    df
    .groupby("PS_PARTKEY")["TOTAL_COST"]
    .sum()
    .reset_index()
    .rename(columns={"TOTAL_COST": "VALUE"})
)

# Filter and sort
total = total[total.VALUE > threshold].sort_values("VALUE", ascending=False)

CPU times: user 67.6 ms, sys: 36 ms, total: 104 ms
Wall time: 104 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q11/rewritten/o4_mini_high/checkpoints/post_cell_3_try_3.pickle

migration speed (bps): 790046267.599747
---------------------------
variables to migrate:
threshold 32
load_partsupp 144
df 48
total 48
DATA_ROOT 80
supplier 48
load_supplier 144
nation 48
STORAGE_OPTS 64
load_nation 144
pd 72
sup 48
tpch_parent 54
ps 48
nat 48
partsupp 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q11/rewritten/o4_mini_high/checkpoints/post_cell_3_try_3.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables ['partsupp']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['supplier']
Future variables ['partsupp']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['nation']
Future variables ['supplier', 'partsupp']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables ['nat', 'total', 'sup', 'threshold', 'df', 'ps']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q11/opt_cell_exec_info_3_try_3.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[3], f)


In [8]:
opt_output = Out.get(4)

In [9]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q11/annotated/checkpoints/post_cell_3.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['ps_supp_merge']
me:  4
trying: ['nation']
me:  3
trying: ['orig_output']
me:  5
trying: ['load_nation']
me:  3
trying: ['supplier_filtered']
me:  4
trying: ['tpch_parent']
me:  0
trying: ['sum_val']
me:  4
trying: ['STORAGE_OPTS']
me:  0
trying: ['ps_supp_n_merge']
me:  4
trying: ['partsupp']
me:  1
trying: ['DATA_ROOT']
me:  0
trying: ['total']
me:  4
trying: ['load_partsupp']
me:  1
trying: ['nation_filtered']
me:  4
trying: ['pd']
me:  0
trying: ['supplier']
me:  2
trying: ['partsupp_filtered']
me:  4
trying: ['load_supplier']
me:  2


Declaring variable tpch_parent
Declaring variable STORAGE_OPTS
Declaring variable DATA_ROOT
Declaring variable pd
Declaring variable partsupp
Declaring variable load_partsupp
Declaring variable supplier
Declaring variable load_supplier
Declaring variable nation
Declaring variable load_nation
Declaring variable ps_supp_merge
Declaring variable supplier_filtered
Declaring variable sum_val
Declaring variable ps_supp_n_merge
Declaring variable total
Declaring variable nation_filtered
Declaring variable partsupp_filtered
Declaring variable orig_output
